In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 97 — Long-Horizon Task Agent

## Goal
Break a **big task** into subtasks, **persist state** to disk
after each subtask, and **resume** later — essential for
tasks that take hours or may be interrupted.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Task decomposition** | LLM breaks big task into subtasks |
| **State persistence** | Save checkpoint to JSON after each step |
| **Resume capability** | Load checkpoint and continue processing |
| **Progress tracking** | Track which subtasks are done |

## Stack
- `langgraph` — stateful graph
- `langchain-ollama` — local LLM
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings, json, time
from pathlib import Path
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOllama(model="qwen3.5:9b", temperature=0)

CHECKPOINT_FILE = "checkpoint_97.json"
print("All imports OK")

## Step 1 — State Schema with Checkpointing

In [ ]:
class LongHorizonState(TypedDict):
    big_task: str
    subtasks: list         # [{"id": 1, "desc": ..., "status": "pending"/"done", "result": ...}]
    current_subtask: int
    completed_results: list
    final_output: str

def save_checkpoint(state: LongHorizonState):
    """Persist state to disk."""
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(dict(state), f, indent=2, default=str)
    print(f"   💾 Checkpoint saved ({state['current_subtask']}/{len(state['subtasks'])} done)")

def load_checkpoint() -> LongHorizonState | None:
    """Load state from disk if it exists."""
    if Path(CHECKPOINT_FILE).exists():
        with open(CHECKPOINT_FILE, "r") as f:
            data = json.load(f)
        print(f"   📂 Checkpoint loaded ({data['current_subtask']}/{len(data['subtasks'])} done)")
        return data
    return None

print("Checkpoint functions defined")

In [ ]:
def decompose_task(state: LongHorizonState) -> LongHorizonState:
    """Break the big task into subtasks."""
    print(f"\n🗂️ Decomposing: {state['big_task'][:60]}...")
    
    # Check for existing checkpoint
    checkpoint = load_checkpoint()
    if checkpoint and checkpoint["big_task"] == state["big_task"]:
        print("   Resuming from checkpoint!")
        return checkpoint
    
    msg = llm.invoke([
        SystemMessage(content="""Break this large task into 4-6 concrete subtasks.
Return a JSON array:
[{"id": 1, "desc": "subtask description"}, ...]
Each subtask should produce a tangible output.
Return ONLY the JSON array."""),
        HumanMessage(content=state["big_task"])
    ])
    
    raw = msg.content
    if "<think>" in raw:
        raw = raw.split("</think>")[-1].strip()
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    
    try:
        start = raw.find("[")
        end = raw.rfind("]") + 1
        subtasks = json.loads(raw[start:end])
    except (json.JSONDecodeError, ValueError):
        subtasks = [
            {"id": 1, "desc": "Research the topic"},
            {"id": 2, "desc": "Outline the key sections"},
            {"id": 3, "desc": "Write the first draft"},
            {"id": 4, "desc": "Review and refine"},
            {"id": 5, "desc": "Final formatting"}
        ]
    
    for st in subtasks:
        st["status"] = "pending"
        st["result"] = ""
    
    for st in subtasks:
        print(f"   📌 Subtask {st['id']}: {st['desc']}")
    
    return {**state, "subtasks": subtasks, "current_subtask": 0, "completed_results": []}

print("Node: decompose_task defined")

In [ ]:
def execute_subtask(state: LongHorizonState) -> LongHorizonState:
    """Execute the current subtask."""
    idx = state["current_subtask"]
    subtask = state["subtasks"][idx]
    
    # Skip if already done (resuming from checkpoint)
    if subtask["status"] == "done":
        print(f"\n⏭️ Subtask {subtask['id']} already done, skipping")
        return {**state, "current_subtask": idx + 1}
    
    print(f"\n⚙️ Executing subtask {subtask['id']}/{len(state['subtasks'])}: {subtask['desc']}")
    
    # Build context from prior results
    prior = ""
    for r in state["completed_results"]:
        prior += f"\n[Subtask {r['id']}]: {r['result'][:200]}\n"
    
    msg = llm.invoke([
        SystemMessage(content=f"""You are working on a large project step by step.
Overall goal: {state['big_task']}

Complete this specific subtask. Be concrete and produce a useful result.
Keep your output to 100-200 words. Do NOT use thinking tags."""),
        HumanMessage(content=f"Subtask: {subtask['desc']}\n\nPrior work:{prior}")
    ])
    
    result = msg.content
    if "<think>" in result:
        result = result.split("</think>")[-1].strip()
    
    # Update subtask
    subtasks = list(state["subtasks"])
    subtasks[idx] = {**subtask, "status": "done", "result": result}
    completed = list(state["completed_results"]) + [{"id": subtask["id"], "result": result}]
    
    new_state = {**state, "subtasks": subtasks, "current_subtask": idx + 1,
                 "completed_results": completed}
    
    # Checkpoint after each subtask
    save_checkpoint(new_state)
    
    return new_state

print("Node: execute_subtask defined")

In [ ]:
def check_progress(state: LongHorizonState) -> str:
    """Route: more subtasks or finalize?"""
    if state["current_subtask"] >= len(state["subtasks"]):
        return "finalize"
    return "execute"

def finalize_task(state: LongHorizonState) -> LongHorizonState:
    """Combine all subtask results into final output."""
    print("\n✅ All subtasks complete! Generating final output...")
    
    all_results = "\n\n".join(
        f"## Subtask {r['id']}\n{r['result']}" for r in state["completed_results"]
    )
    
    msg = llm.invoke([
        SystemMessage(content="""Combine all subtask results into a cohesive final output.
Synthesize, don't just concatenate. 200-300 words. No thinking tags."""),
        HumanMessage(content=f"Task: {state['big_task']}\n\nSubtask Results:\n{all_results}")
    ])
    
    final = msg.content
    if "<think>" in final:
        final = final.split("</think>")[-1].strip()
    
    # Clean up checkpoint
    if Path(CHECKPOINT_FILE).exists():
        Path(CHECKPOINT_FILE).unlink()
        print("   🗑️ Checkpoint file removed (task complete)")
    
    return {**state, "final_output": final}

print("Nodes: check_progress, finalize defined")

In [ ]:
# Build the graph
graph = StateGraph(LongHorizonState)

graph.add_node("decompose", decompose_task)
graph.add_node("execute", execute_subtask)
graph.add_node("finalize", finalize_task)

graph.set_entry_point("decompose")
graph.add_edge("decompose", "execute")
graph.add_conditional_edges("execute", check_progress, {
    "execute": "execute",
    "finalize": "finalize"
})
graph.add_edge("finalize", END)

long_horizon_app = graph.compile()
print("Long-horizon task graph compiled!")

In [ ]:
# Run the long-horizon agent
result = long_horizon_app.invoke({
    "big_task": "Write a comprehensive market analysis report for a startup entering the AI developer tools space, covering market size, competitors, differentiation strategy, go-to-market plan, and financial projections.",
    "subtasks": [],
    "current_subtask": 0,
    "completed_results": [],
    "final_output": ""
})

print("\n" + "=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result["final_output"])

print(f"\nSubtasks completed: {len(result['completed_results'])}")
for r in result["completed_results"]:
    print(f"  ✅ Subtask {r['id']}: {r['result'][:60]}...")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Decomposition** | LLM breaks task into 4-6 subtasks |
| **Persistence** | `save_checkpoint()` after each subtask |
| **Resume** | `load_checkpoint()` skips completed subtasks |
| **Progress loop** | Execute → check → execute until done |
| **Synthesis** | Final node combines all subtask outputs |

## Architecture
```
Decompose → Execute Subtask → [more?]
  ↑ (load checkpoint)  ↓ (save checkpoint)
                    File: checkpoint.json
              ↓ YES        ↓ NO
         Execute Next   Finalize → END
```

## Why Persistence Matters
- LLM API calls can timeout or fail
- Long tasks may span multiple sessions
- Users may want to pause and resume
- Crash recovery without re-doing work

## Extending This Project
- Database-backed checkpoints instead of JSON
- Parallel subtask execution for independent steps
- User confirmation between phases
- Priority-based subtask ordering